# Booking cancellation prediction — final training notebook

This notebook mirrors the modular training pipeline in `src/backend/main.py`.

It runs:
1. load frozen training data from PostgreSQL
2. offline preprocessing (fit + transform train; transform val/test)
3. model training (default: XGBoost)
4. optional hyperparameter / threshold tuning
5. evaluation on the held-out test set
6. save model + prediction artifacts

Assumes data has already been ingested into serving (`ingest_csv`).


## Config

Defaults match `run_main` / `backend.main`:
- `MODEL_TYPE = "xgb"`
- `HYPERPARAMETER_TUNING = True`
- `THRESHOLD_TUNING = True`

Set those flags to `False` for a faster smoke run.


In [1]:
from datetime import datetime, timedelta

from joblib import dump
from dotenv import load_dotenv

from database import create_datamanager
from backend.preprocessing import run_offline_preprocessing, full_custom_transform
from backend.model import initialize_model, apply_best_threshold, xgb_scale_pos_weight
from backend.tuning import custom_hyperparameter_tuning, custom_temporal_cv, find_best_threshold
from backend.evaluation import evaluate_custom_predictions
from artifacts.artifacts import (
    save_prediction_artifacts,
    save_threshold_artifacts,
    load_preprocessing_artifacts,
)
from config.settings import (
    LOAD_OFFLINE_DATA_QUERY,
    MODEL_PARAMS,
    PARAM_TUNING_GRID,
    GRID_SEARCH_PARAMS,
    MODEL_PATH,
    MIN_ACCEPTED_PRECISION,
    ARTIFACTS_PATH,
)

load_dotenv()

MODEL_TYPE = "xgb"  # or "random_forest"
HYPERPARAMETER_TUNING = True
THRESHOLD_TUNING = True

print("MODEL_TYPE:", MODEL_TYPE)
print("HYPERPARAMETER_TUNING:", HYPERPARAMETER_TUNING)
print("THRESHOLD_TUNING:", THRESHOLD_TUNING)
print("MIN_ACCEPTED_PRECISION:", MIN_ACCEPTED_PRECISION)

MODEL_TYPE: xgb
HYPERPARAMETER_TUNING: True
THRESHOLD_TUNING: True
MIN_ACCEPTED_PRECISION: 0.875


## 1. Load offline data

Same query and 90-day cutoff as `backend.main`: frozen rooms only (`is_frozen = true`), booked before the train/inference cutoff.


In [2]:
train_inference_time_cutoff = datetime.now() - timedelta(days=90)
datamanager = create_datamanager()
df = datamanager.load_query(
    LOAD_OFFLINE_DATA_QUERY,
    params={"time_cutoff": train_inference_time_cutoff},
)

print("cutoff:", train_inference_time_cutoff)
print("rows loaded:", len(df))
print("columns:", list(df.columns))
df.head()

cutoff: 2026-04-09 19:57:27.289060
rows loaded: 18894
columns: ['resid', 'ref', 'book_owner', 'booked on', 'property_name', 'arrival_date', 'departure_date', 'nights', 'custid', 'customer_notes', 'cust_country', 'date_cancelled', 'status', 'pax', 'unit_code', 'room_code', 'room_amount', 'extras_amount', 'tot_amount', 'pay_amount', 'madeby', 'voucher', 'balance']


,resid,ref,book_owner,booked on,property_name,arrival_date,departure_date,nights,custid,customer_notes,...,pax,unit_code,room_code,room_amount,extras_amount,tot_amount,pay_amount,madeby,voucher,balance
0,26952116,13959,136bealey,2025-04-30 16:34:26.822138,Abbey Motor Lodge,2025-05-01 14:00:00,2025-05-05 10:00:00,4,17074527.0,NaN,...,2,Room 06,Standard Studio,497.26,0.0,497.26,497.26,136bealeybookcom,BCOM-4666975805,0.0
1,20328851,8980,136bealey,2021-08-08 15:57:13.497693,Abbey Motor Lodge,2021-09-20 14:00:00,2021-09-21 10:00:00,1,12458953.0,NaN,...,2,Dummy3,Standard Studio,81.56,0.0,81.56,81.56,136bealeyagoda,AGO-290919303,0.0
2,27225605,14572,136bealey,2025-07-19 10:27:45.556682,Abbey Motor Lodge,2025-08-22 14:00:00,2025-08-24 10:00:00,2,17266670.0,NaN,...,2,Room 10,Standard Studio,211.14,0.0,211.14,211.14,136bealeybookcom,BCOM-5332975403,0.0
3,27898943,16689,136bealey,2026-01-03 13:52:56.723339,Abbey Motor Lodge,2026-01-11 14:00:00,2026-01-14 10:00:00,3,17674374.0,NaN,...,1,Room 04,Executive Twin Studio,300.00,0.0,300.00,300.00,luo,NaN,0.0
4,17601975,4937,136bealey,2019-11-11 00:00:00.000000,Abbey Motor Lodge,2020-01-31 14:00:00,2020-02-01 10:00:00,1,10622756.0,NaN,...,2,Room 01,dummy,98.60,0.0,98.60,98.60,Import,"10,174",0.0


## 2. Offline preprocessing

Matches `run_offline_preprocessing` + `full_custom_transform`:
- clean + room-level features
- temporal booking split (train / val / test)
- fit room-code lookup, historical rate lookup, and `madeby` OHE on train
- transform val/test with the fitted artifacts


In [3]:
X_train, Y_train, full_val, full_test = run_offline_preprocessing(df)
room_code_lookup, room_rate_lookup, OH_encoder = load_preprocessing_artifacts()

X_val, Y_val = full_custom_transform(
    full_val, room_code_lookup, room_rate_lookup, OH_encoder, is_offline=True
)
X_test, Y_test = full_custom_transform(
    full_test, room_code_lookup, room_rate_lookup, OH_encoder, is_offline=True
)

print("X_train:", X_train.shape, "Y_train cancel rate:", float(Y_train.mean()))
print("X_val:", X_val.shape, "Y_val cancel rate:", float(Y_val.mean()))
print("X_test:", X_test.shape, "Y_test cancel rate:", float(Y_test.mean()))
print("feature columns:")
list(X_train.columns)

[2026-07-08 19:57:27,624] INFO artifacts.py:24 - Artifact saved to C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\room_code_lookup.joblib
[2026-07-08 19:57:27,626] INFO artifacts.py:24 - Artifact saved to C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\historical_room_rates.joblib
[2026-07-08 19:57:27,627] INFO artifacts.py:24 - Artifact saved to C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\madeby_encoder.joblib
[2026-07-08 19:57:27,628] INFO artifacts.py:46 - Preprocessing artifacts saved
[2026-07-08 19:57:27,629] INFO artifacts.py:29 - Joblib artifacts loaded from C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\room_code_lookup.joblib
[2026-07-08 19:57:27,630] INFO artifacts.py:29 - Joblib artifacts loaded from C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\historical_room_rates.joblib
[2026-07-08 19:57:27,631] INFO artifacts.py:29 - Joblib artifacts loaded from C:\Users

X_train: (10405, 15) Y_train cancel rate: 0.18250840941854876
X_val: (2506, 15) Y_val cancel rate: 0.19792498004788509
X_test: (3185, 15) Y_test cancel rate: 0.13971742543171115
feature columns:


['is_136_motel',
 'has_customer_notes',
 'is_domestic',
 'has_voucher',
 'total_rooms',
 'total_guests',
 'total_room_revenue',
 'average_room_rate',
 'lead_time_days',
 'average_price_pp',
 'madeby_136bealey',
 'madeby_136bealeyagoda',
 'madeby_136bealeybookcom',
 'madeby_136bealeyexpedia',
 'madeby_Import']

## 3. Train model

Default path matches production defaults (`xgb`).
If `HYPERPARAMETER_TUNING` is True, uses temporal CV + grid search from `backend.tuning`.


In [4]:
model_params = {**MODEL_PARAMS[MODEL_TYPE]}
if MODEL_TYPE == "xgb":
    model_params["scale_pos_weight"] = xgb_scale_pos_weight(Y_train)

model = initialize_model(MODEL_TYPE, model_params)

if HYPERPARAMETER_TUNING:
    cv_results = custom_temporal_cv(X_train)
    tuned_model = custom_hyperparameter_tuning(
        X_train,
        Y_train,
        PARAM_TUNING_GRID[MODEL_TYPE],
        estimator=model,
        cv_splits=cv_results,
        grid_search_params=GRID_SEARCH_PARAMS,
    )
else:
    tuned_model = model.fit(X_train, Y_train)

print("trained model:", type(tuned_model).__name__)
print("params:", model_params)

trained model: XGBClassifier
params: {'n_jobs': -2, 'random_state': 67, 'scale_pos_weight': np.float64(4.479199578725645)}


## 4. Threshold tuning + predictions

If threshold tuning is enabled:
- build PR curve on validation probabilities
- choose best threshold under `MIN_ACCEPTED_PRECISION`
- apply that threshold to test set

Otherwise:
- use sklearn default `predict`
- clear `threshold_artifacts.json`


In [5]:
if THRESHOLD_TUNING:
    probabilities = tuned_model.predict_proba(X_val)
    best_threshold, precision_vals, recall_vals, threshold_vals = find_best_threshold(
        probabilities, Y_val
    )
    min_accepted_precision = MIN_ACCEPTED_PRECISION
    save_threshold_artifacts(precision_vals, recall_vals, threshold_vals)
    custom_predictions, confidence = apply_best_threshold(
        tuned_model, X_test, best_threshold
    )
else:
    (ARTIFACTS_PATH / "threshold_artifacts.json").unlink(missing_ok=True)
    best_threshold, min_accepted_precision = 0.5, 0.0
    custom_predictions = tuned_model.predict(X_test)
    confidence = tuned_model.predict_proba(X_test)[:, 1]
    print("Threshold tuning skipped; using default threshold 0.5")

print("best_threshold:", best_threshold)
print("min_accepted_precision:", min_accepted_precision)
print("positive predictions:", int(custom_predictions.sum()), "/", len(custom_predictions))

[2026-07-08 19:57:39,570] INFO artifacts.py:81 - Precision, recall, and threshold values have the same length and correct type
[2026-07-08 19:57:39,580] INFO artifacts.py:64 - Artifacts saved to C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\threshold_artifacts.json
[2026-07-08 19:57:39,581] INFO artifacts.py:90 - Threshold artifacts saved


best_threshold: 0.68293977
min_accepted_precision: 0.875
positive predictions: 504 / 3185


## 5. Evaluate / finalize

Same metrics as `evaluate_custom_predictions`, then save model + prediction artifacts exactly like `backend.main`.


In [6]:
f1_score, recall_score, precision_score, confusion_matrix = evaluate_custom_predictions(
    Y_test, custom_predictions
)

print("=== Test evaluation ===")
print(f"weighted F1: {f1_score}")
print(f"precision:   {precision_score:.4f}")
print(f"recall:      {recall_score:.4f}")
print("confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(tuned_model, MODEL_PATH)
save_prediction_artifacts(best_threshold, min_accepted_precision, train_inference_time_cutoff)

print("\nSaved:")
print(" -", MODEL_PATH)
print(" -", ARTIFACTS_PATH / "prediction_artifacts.json")
if THRESHOLD_TUNING:
    print(" -", ARTIFACTS_PATH / "threshold_artifacts.json")

[2026-07-08 19:57:39,605] INFO artifacts.py:64 - Artifacts saved to C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\prediction_artifacts.json
[2026-07-08 19:57:39,606] INFO artifacts.py:105 - Prediction artifacts saved


=== Test evaluation ===
weighted F1: 0.961
precision:   0.8135
recall:      0.9213
confusion matrix [[TN, FP], [FN, TP]]:
[[2646, 94], [35, 410]]

Saved:
 - C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\final_model.joblib
 - C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\prediction_artifacts.json
 - C:\Users\jocac\Projects\booking-cancellation-prediction\src\artifacts\threshold_artifacts.json


## Notes

- This notebook is the training path only (through model finalization / evaluation).
- Serving / Streamlit inference is handled by `app.py` + `streamlit_app.py`.
- After re-running this notebook (or `run_main`), restart the FastAPI server so it reloads artifacts.
